In [ ]:
from os.path import join, dirname
import os
import sys
import gc
import logging
import argparse
from copy import deepcopy
import numpy as np
import torchshow as ts
import torchvision
from einops import rearrange, repeat
from matplotlib import pyplot as plt

import torch
from torch import autocast
from torch.cuda.amp import GradScaler
import torch.nn.functional as F
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.autograd import grad, Variable

sys.path.append(join(dirname(dirname(os.getcwd()))))
sys.path.append(os.getcwd())

from memoryPerceiver.memory_perceiver.main.configuration import cf
from memoryPerceiver.memory_perceiver.core.model_wrapper import create_memories, reset_memories
from memoryPerceiver.memory_perceiver.core.distributed import setup_distributed, average_gradients, scatter_dummy_src, gather_dummy_src
from memoryPerceiver.memory_perceiver.core.miscellaneous import custom_random, print_cuda_tensors
import memoryPerceiver.memory_perceiver.isaac.trainer as tr
from memoryPerceiver.memory_perceiver.isaac.plot import Plotter
from memoryPerceiver.memory_perceiver.isaac.agent import ActorCritic, ActorCriticSimple, ActorCriticPerceiver, create_agent

from memoryPerceiver.memory_perceiver.core.model_net import PerceiverIO
from memoryPerceiver.memory_perceiver.core.model_wrapper import create_multimodal_perceiver, output_projection, input_embedding
from memoryPerceiver.memory_perceiver.core.rnn import SimpleRNN
from memoryPerceiver.memory_perceiver.core.rnn import SimpleRNN
from memoryPerceiver.memory_perceiver.core.rtrl import UORO, RTRL, RTRL_partial

%load_ext autoreload
%autoreload 2
device = 'cuda:1'

In [ ]:
# Function to print all tensors on CUDA
def print_cuda_tensors():
    # Check if CUDA is available
    if not torch.cuda.is_available():
        print("CUDA is not available.")
        return
    
    # Get the list of all tensors
    all_tensors = [obj for obj in gc.get_objects() if isinstance(obj, torch.Tensor)]
    
    # Filter for tensors on CUDA
    cuda_tensors = [t for t in all_tensors if t.is_cuda]
    cuda_tensors.sort(key=lambda t: t.numel(), reverse=True)
    
    # Print details about each tensor
    for tensor in cuda_tensors:
        print(f"Tensor: {tensor.requires_grad}, Size: {tensor.size()}, Device: {tensor.device}, "
              f"Element Size: {tensor.element_size()}, Numel: {tensor.numel()}, "
              f"Memory: {tensor.numel() * tensor.element_size() / (1024 ** 2):.2f} MB")

# Import garbage collection to find all objects
import gc

# Execute the function
print_cuda_tensors()